In [1]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor
import glob, os
import time
import sys
sys.path.append('..')
from src.preprocessing import FeatureExtractor

In [2]:
folder_path = r'..\data\raw'
file_type = '/*csv'

files = glob.glob(folder_path + file_type)

latest_file = max(files, key=os.path.getctime)

time = latest_file[12:].split(sep="_")

access_time = pd.Timestamp(
    year=int(time[0][0:4]),
    month=int(time[0][4:6]),
    day=int(time[0][6:8]),
    hour=int(time[1][0:2]),
    minute=int(time[1][2:4]),
    tz='Europe/Moscow'
)

df = pd.read_csv(latest_file)

In [3]:
y = df["price_byn"]
X = df.drop(["price_byn"], axis=1)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42)

In [9]:
from sklearn.model_selection import cross_val_score, cross_validate
import numpy as np

categorical_features = [
    "brand",
    "processor",
    "rom_type",
    "os", 
    "videocard",
    "videocard_brand",
    "region",
    "matrix_type",
    "display_resolution",
    "ram_type"
]

cb_pipeline = Pipeline([
    ('extractor', FeatureExtractor(access_time)),
    ('regressor', CatBoostRegressor(
        iterations=1000,
        learning_rate=0.03,
        depth=7,
        loss_function='RMSE',
        early_stopping_rounds=50,
        random_seed=42,
        verbose=100,
        cat_features=categorical_features
    ))
])

In [12]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error

results = {}

def evaluate(name, pipeline, X_train, y_train, X_test, y_test, cv=5):
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    r2   = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)

    cv_scores = cross_validate(
        pipeline, X_train, y_train, cv=cv,
        scoring={"r2": "r2", "rmse": "neg_root_mean_squared_error", "mae": "neg_mean_absolute_error"},
        return_train_score=False,
    )

    results[name] = {
        "R2 (test)":       r2,
        "RMSE (test)":     rmse,
        "MAE (test)":      mae,
        "R2 (cv mean)":    cv_scores["test_r2"].mean(),
        "R2 (cv std)":     cv_scores["test_r2"].std(),
        "RMSE (cv mean)": -cv_scores["test_rmse"].mean(),
        "MAE (cv mean)":  -cv_scores["test_mae"].mean(),
    }

def show_metrics():
    df = pd.DataFrame(results).T
    higher_is_better = ["R2 (test)", "R2 (cv mean)"]
    lower_is_better  = ["RMSE (test)", "MAE (test)", "RMSE (cv mean)", "MAE (cv mean)", "R2 (cv std)"]
    return (
        df.style
        .highlight_max(subset=[c for c in higher_is_better if c in df.columns], color="#32CD32", axis=0)
        .highlight_min(subset=[c for c in lower_is_better  if c in df.columns], color="#32CD32", axis=0)
        .format("{:.4f}")
    )

In [11]:
evaluate("CatBoost", cb_pipeline, X_train, y_train, X_test, y_test)

0:	learn: 1970.0142628	total: 67ms	remaining: 1m 6s
100:	learn: 1059.2496985	total: 8.66s	remaining: 1m 17s
200:	learn: 966.1942227	total: 17s	remaining: 1m 7s
300:	learn: 915.7283622	total: 25.7s	remaining: 59.7s
400:	learn: 884.6283845	total: 34.2s	remaining: 51.1s
500:	learn: 860.8826839	total: 42.9s	remaining: 42.7s
600:	learn: 835.9587460	total: 51.7s	remaining: 34.3s
700:	learn: 816.3434532	total: 1m	remaining: 25.6s
800:	learn: 799.7255093	total: 1m 8s	remaining: 17.1s
900:	learn: 782.0886313	total: 1m 17s	remaining: 8.49s
999:	learn: 765.7517872	total: 1m 26s	remaining: 0us
0:	learn: 1953.6853609	total: 49.7ms	remaining: 49.7s
100:	learn: 1065.5262248	total: 8.47s	remaining: 1m 15s
200:	learn: 960.1602298	total: 16.7s	remaining: 1m 6s
300:	learn: 907.9733678	total: 25.3s	remaining: 58.7s
400:	learn: 864.7514500	total: 33.2s	remaining: 49.7s
500:	learn: 833.5077805	total: 41.5s	remaining: 41.3s
600:	learn: 808.4601201	total: 49.5s	remaining: 32.8s
700:	learn: 787.4106788	total: 

,R2 (test),RMSE (test),MAE (test),R2 (cv mean),R2 (cv std),RMSE (cv mean),MAE (cv mean)
CatBoost,0.7409,992.3829,511.8648,0.7362,0.0318,1026.3186,522.3111


In [ ]:
y_pred = cb_pipeline.pred(x_train)

In [14]:
show_metrics()